# LeetCode 1870 — Minimum Speed to Arrive on Time bằng Python

Link đề: https://leetcode.com/problems/minimum-speed-to-arrive-on-time/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search

Skeleton LeetCode:

```python
class Solution:
    def minSpeedOnTime(self, dist: List[int], hour: float) -> int:
        ...
```

## 1. Mục tiêu học

1. Nhận dạng bài **binary search trên đáp án**: hàm "đến kịp với tốc độ `v`?" **đơn điệu** theo `v` (nhanh hơn không bao giờ chậm hơn).
2. Mô hình hóa thời gian đúng: mọi chuyến **trừ chuyến cuối** bị làm tròn lên số nguyên (`⌈d/v⌉`), chuyến cuối giữ phần lẻ `d/v`.
3. Tránh lỗi **số thực dấu phẩy động**: đổi `hour` (tối đa 2 chữ số thập phân) thành số nguyên **phần trăm giờ** và so sánh hoàn toàn bằng số nguyên.
4. Xác định **điều kiện bất khả thi** (`hour ≤ n − 1`) và **cận trên** của tốc độ (`10⁷`, hoặc cận chặt hơn tính từ dữ liệu).
5. Kiểm chứng bằng quét tuyến tính và số hữu tỉ `Fraction`.

## 2. Đề bài & ví dụ

Cho số thực `hour` — thời gian bạn có để đến văn phòng. Bạn phải đi lần lượt `n` chuyến tàu; mảng số nguyên `dist` độ dài `n`, `dist[i]` là quãng đường (km) của chuyến thứ `i`.

Mỗi chuyến tàu chỉ khởi hành vào **giờ nguyên**, nên có thể phải chờ giữa các chuyến. Ví dụ chuyến 1 mất 1.5 giờ thì phải chờ thêm 0.5 giờ, chuyến 2 khởi hành ở mốc 2 giờ.

Trả về **tốc độ nguyên dương nhỏ nhất** (km/h) mà mọi chuyến cùng chạy để bạn đến **đúng hạn**, hoặc `−1` nếu không thể. Đề đảm bảo đáp án không vượt `10⁷` và `hour` có tối đa hai chữ số sau dấu phẩy.

**Ví dụ 1:** `dist = [1,3,2], hour = 6` → `1` (1 + 3 + 2 = 6 giờ, không phải chờ).

**Ví dụ 2:** `dist = [1,3,2], hour = 2.7` → `3` (1/3 giờ, chờ đến mốc 1; 1 giờ đến mốc 2; 2/3 giờ ⇒ đến lúc 2.667).

**Ví dụ 3:** `dist = [1,3,2], hour = 1.9` → `−1` (chuyến thứ ba sớm nhất khởi hành ở mốc 2).

**Ràng buộc:** `1 ≤ n ≤ 10⁵`; `1 ≤ dist[i] ≤ 10⁵`; `1 ≤ hour ≤ 10⁹`; `hour` có tối đa hai chữ số thập phân.

In [1]:
import math
import random
import time
import bisect
from fractions import Fraction
from typing import List

examples = [(([1, 3, 2], 6), 1), (([1, 3, 2], 2.7), 3), (([1, 3, 2], 1.9), -1)]
for (dist, hour), exp in examples:
    print(f"dist={dist}, hour={hour} -> kỳ vọng {exp}")

dist=[1, 3, 2], hour=6 -> kỳ vọng 1
dist=[1, 3, 2], hour=2.7 -> kỳ vọng 3
dist=[1, 3, 2], hour=1.9 -> kỳ vọng -1


## 3. Kiến thức nền

### 3.1 Thời gian đi với tốc độ `v`

Chuyến `i < n−1` chiếm `⌈dist[i] / v⌉` giờ (đi xong rồi chờ tới giờ nguyên); chuyến cuối chiếm đúng `dist[n−1] / v` (đến nơi là xong, không chờ).

`T(v) = Σ_{i<n−1} ⌈dist[i]/v⌉ + dist[n−1]/v`.

### 3.2 Chia làm tròn lên bằng số nguyên

`⌈a / b⌉ = (a + b − 1) // b = −(−a // b)` với `a, b > 0` — không dùng số thực.

In [2]:
for a, b in [(1, 3), (3, 3), (7, 2), (10 ** 5, 7)]:
    print(f"ceil({a}/{b}) = {(a + b - 1) // b} = {-(-a // b)} = {math.ceil(a / b)}")

ceil(1/3) = 1 = 1 = 1
ceil(3/3) = 1 = 1 = 1
ceil(7/2) = 4 = 4 = 4
ceil(100000/7) = 14286 = 14286 = 14286


### 3.3 Số thực và hai chữ số thập phân

`2.7`, `1.14` không biểu diễn chính xác trong nhị phân. Cộng/chia số thực rồi so sánh `≤` có thể sai ở **đúng biên**. Vì `hour` có tối đa hai chữ số thập phân, đặt `H = round(hour · 100)` (số nguyên) và nhân cả hai vế bất phương trình với `100·v`:

`T(v) ≤ hour  ⇔  100·v·S + 100·dist[n−1] ≤ H·v`, với `S = Σ_{i<n−1} ⌈dist[i]/v⌉`.

In [3]:
print(repr(1 + 14 / 100), "<=", repr(1.14), "?", 1 + 14 / 100 <= 1.14)
print("Số nguyên:", 100 * 100 * 1 + 100 * 14, "<=", round(1.14 * 100) * 100, "?", 100 * 100 * 1 + 100 * 14 <= round(1.14 * 100) * 100)

1.1400000000000001 <= 1.14 ? False
Số nguyên: 11400 <= 11400 ? True


### 3.4 Binary search trên đáp án

Nếu `ok(v)` đơn điệu (sai…sai, đúng…đúng), tìm `v` nhỏ nhất với `ok(v)` đúng trên `[lo, hi]` bằng `O(log(hi − lo))` lần gọi `ok`.

In [4]:
def first_true(lo, hi, ok):
    # giả định ok(hi) đúng; trả v nhỏ nhất trong [lo, hi] với ok(v)
    while lo < hi:
        mid = (lo + hi) // 2
        if ok(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

print(first_true(1, 100, lambda v: v * v >= 50))

8


### 3.5 Vì sao phải `round(hour * 100)` chứ không phải `int(hour * 100)`

Phép nhân số thực có thể cho kết quả **hơi nhỏ hơn** số nguyên mong muốn; `int` cắt bỏ phần lẻ nên mất một đơn vị. `round` đưa về số nguyên gần nhất — đúng vì `hour` có tối đa hai chữ số thập phân. Thống kê dưới đây quét mọi `hour` dạng `k/100` với `k` từ 100 đến 20000.

In [5]:
bad_int = [k for k in range(100, 20001) if int((k / 100) * 100) != k]
bad_round = [k for k in range(100, 20001) if round((k / 100) * 100) != k]
print(f"int() sai ở {len(bad_int)} giá trị, ví dụ hour = {[b / 100 for b in bad_int[:5]]}")
print(f"round() sai ở {len(bad_round)} giá trị")
print("Ví dụ cụ thể:", repr(2.01 * 100), "-> int:", int(2.01 * 100), "| round:", round(2.01 * 100))

int() sai ở 1142 giá trị, ví dụ hour = [1.13, 1.14, 1.15, 1.16, 2.01]
round() sai ở 0 giá trị
Ví dụ cụ thể: 200.99999999999997 -> int: 200 | round: 201


## 4. Ý tưởng nền tảng / trực giác

`T(v)` **giảm không tăng** khi `v` tăng (mỗi số hạng `⌈d/v⌉`, `d/v` đều không tăng). Vậy tập tốc độ đến kịp là một **tia** `[v*, ∞)` — tìm `v*` bằng binary search.

**Khi nào không thể?** Dù tốc độ lớn đến đâu, mỗi chuyến trừ chuyến cuối vẫn chiếm ít nhất 1 giờ ⇒ `T(v) > n − 1` luôn. Vậy nếu `hour ≤ n − 1` ⇒ `−1`. Đề còn đảm bảo nếu có đáp án thì `≤ 10⁷`, nên chỉ cần kiểm tra `ok(10⁷)`.

Tính tay **Ví dụ 2** (`hour = 2.7`):

| v | ⌈1/v⌉ | ⌈3/v⌉ | 2/v | T(v) | ≤ 2.7? |
|---|---|---|---|---|---|
| 1 | 1 | 3 | 2 | 6 | ✗ |
| 2 | 1 | 2 | 1 | 4 | ✗ |
| 3 | 1 | 1 | 0.667 | 2.667 | **✓** |
| 4 | 1 | 1 | 0.5 | 2.5 | ✓ |

In [6]:
def T_exact(dist, v):
    return sum(Fraction(-(-d // v)) for d in dist[:-1]) + Fraction(dist[-1], v)

dist = [1, 3, 2]
for v in range(1, 6):
    t = T_exact(dist, v)
    print(f"v={v}: T = {t} ≈ {float(t):.3f} | ≤ 2.7: {t <= Fraction(27, 10)} | ≤ 6: {t <= 6} | ≤ 1.9: {t <= Fraction(19, 10)}")

v=1: T = 6 ≈ 6.000 | ≤ 2.7: False | ≤ 6: True | ≤ 1.9: False
v=2: T = 4 ≈ 4.000 | ≤ 2.7: False | ≤ 6: True | ≤ 1.9: False
v=3: T = 8/3 ≈ 2.667 | ≤ 2.7: True | ≤ 6: True | ≤ 1.9: False
v=4: T = 5/2 ≈ 2.500 | ≤ 2.7: True | ≤ 6: True | ≤ 1.9: False
v=5: T = 12/5 ≈ 2.400 | ≤ 2.7: True | ≤ 6: True | ≤ 1.9: False


## 5. Cách 1 — Brute Force / baseline: thử lần lượt `v = 1, 2, 3, …`

Dùng `Fraction` để tính `T(v)` chính xác; dừng ở `v` đầu tiên đến kịp. Giới hạn `v ≤ 10⁷` (theo đề), và trả `−1` sớm nếu `hour ≤ n − 1`.

In [7]:
def minspeed_bruteforce(dist, hour, limit=10 ** 7):
    h = Fraction(round(hour * 100), 100)
    if h <= len(dist) - 1:
        return -1
    for v in range(1, limit + 1):
        if T_exact(dist, v) <= h:
            return v
    return -1

for (dist, hour), exp in examples:
    got = minspeed_bruteforce(dist, hour); print(got, "kỳ vọng", exp); assert got == exp

1 kỳ vọng 1
3 kỳ vọng 3
-1 kỳ vọng -1


**Độ phức tạp:** `O(v* · n)` — tệ nhất `10⁷ · 10⁵ = 10¹²` ⇒ **TLE**. Chỉ dùng cho dữ liệu nhỏ (đáp án nhỏ).

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (đơn điệu):** `ok(v) = [T(v) ≤ hour]` đơn điệu ⇒ binary search trên `v ∈ [1, 10⁷]`, mỗi lần kiểm tra `O(n)` ⇒ `O(n log 10⁷) ≈ 24n`.

**Insight 2 (số nguyên tuyệt đối):** kiểm tra `100·v·S + 100·d_last ≤ H·v` với `H = round(hour·100)` — không có sai số.

**Insight 3 (bất khả thi):** `−1` ⇔ `H ≤ 100·(n − 1)` (với giới hạn `10⁷` của đề, điều kiện này tương đương `ok(10⁷)` sai).

**Insight 4 (cận chặt hơn, tùy chọn):** với `v ≥ max(dist)`, mọi `⌈d/v⌉ = 1` ⇒ `T(v) = (n−1) + d_last/v`. Nên `v*` `≤ max(max(dist), ⌈100·d_last / (H − 100(n−1))⌉)`.

**Invariant của binary search:** `ok(hi)` luôn đúng, `ok(lo − 1)` luôn sai (hoặc `lo = 1`).

### 6.1 Hình dạng hàm `T(v)`: bậc thang giảm dần

In `T(v)` cho vài `v` với `dist = [7, 5, 9]`: phần nguyên `Σ ⌈d/v⌉` giảm theo **bậc thang** và chạm sàn `n − 1 = 2` khi `v ≥ max(dist[:-1])`; phần lẻ `d_last/v` tiếp tục giảm dần về 0 nhưng không bao giờ bằng 0. Đó là lý do `hour ≤ n − 1` là bất khả thi và đáp án có thể rất lớn khi `hour` chỉ nhỉnh hơn `n − 1` một chút.

In [8]:
dist = [7, 5, 9]
for v in (1, 2, 3, 4, 5, 7, 9, 18, 90, 900):
    S = sum(-(-d // v) for d in dist[:-1]); t = T_exact(dist, v)
    print(f"v={v:>4}: Σceil = {S:>2}, d_last/v = {float(Fraction(dist[-1], v)):.4f}, T = {float(t):.4f}")

v=   1: Σceil = 12, d_last/v = 9.0000, T = 21.0000
v=   2: Σceil =  7, d_last/v = 4.5000, T = 11.5000
v=   3: Σceil =  5, d_last/v = 3.0000, T = 8.0000
v=   4: Σceil =  4, d_last/v = 2.2500, T = 6.2500
v=   5: Σceil =  3, d_last/v = 1.8000, T = 4.8000
v=   7: Σceil =  2, d_last/v = 1.2857, T = 3.2857
v=   9: Σceil =  2, d_last/v = 1.0000, T = 3.0000
v=  18: Σceil =  2, d_last/v = 0.5000, T = 2.5000
v=  90: Σceil =  2, d_last/v = 0.1000, T = 2.1000
v= 900: Σceil =  2, d_last/v = 0.0100, T = 2.0100


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Binary search với kiểm tra bằng số nguyên (bản nộp)

In [9]:
def make_ok(dist, hour):
    H = round(hour * 100)
    head, last = dist[:-1], dist[-1]
    def ok(v):
        S = sum((d + v - 1) // v for d in head)
        return 100 * S * v + 100 * last <= H * v
    return ok, H

def minspeed_bs(dist, hour):
    ok, H = make_ok(dist, hour)
    if H <= 100 * (len(dist) - 1):
        return -1
    hi = 10 ** 7
    if not ok(hi):
        return -1
    return first_true(1, hi, ok)

print([minspeed_bs(*a) for a, _ in examples])

[1, 3, -1]


**Độ phức tạp Cách 2:** `O(n log M)` với `M = 10⁷` (≈ 24 lần kiểm tra × `n`). Space `O(n)` cho lát cắt `dist[:-1]` (có thể `O(1)`).

### 7.2 Cách 3 — Binary search với cận trên chặt (Insight 4)

In [10]:
def minspeed_tight(dist, hour):
    ok, H = make_ok(dist, hour)
    n = len(dist)
    slack = H - 100 * (n - 1)                  # (hour - (n-1)) * 100
    if slack <= 0:
        return -1
    hi = max(max(dist), -(-100 * dist[-1] // slack))
    assert ok(hi)
    return first_true(1, hi, ok)

print([minspeed_tight(*a) for a, _ in examples])

[1, 3, -1]


Tại `v = hi ≥ max(dist)`: mọi chuyến đầu chiếm đúng 1 giờ, chuyến cuối `d_last/hi ≤ slack/100` ⇒ đến kịp. Cận này không dựa vào bảo đảm "`≤ 10⁷`" của đề — luôn đúng.

**Độ phức tạp Cách 3:** `O(n log hi)` với `hi ≤ 10⁷`.

### 7.3 Cách 4 — `bisect` với `key` (Python 3.10+)

`bisect_left(range(1, hi + 1), True, key=ok)` tìm vị trí đầu tiên `ok` đúng — gọn, cùng độ phức tạp.

In [11]:
def minspeed_bisect(dist, hour):
    ok, H = make_ok(dist, hour)
    if H <= 100 * (len(dist) - 1):
        return -1
    hi = 10 ** 7
    if not ok(hi):
        return -1
    return 1 + bisect.bisect_left(range(1, hi + 1), True, key=ok)

print([minspeed_bisect(*a) for a, _ in examples])

[1, 3, -1]


`range` hỗ trợ truy cập chỉ số `O(1)` nên `bisect` hoạt động mà không tạo list `10⁷` phần tử.

## 8. TRACE từng bước

Trace binary search của Cách 2 trên **Ví dụ 2** (`hour = 2.7`, `H = 270`):

In [12]:
def trace_bs(dist, hour, hi=10 ** 7):
    ok, H = make_ok(dist, hour); lo = 1; step = 0
    while lo < hi:
        mid = (lo + hi) // 2; r = ok(mid); step += 1
        S = sum((d + mid - 1) // mid for d in dist[:-1])
        if step <= 6 or hi - lo < 8:
            print(f"bước {step:>2}: lo={lo:<8} hi={hi:<8} mid={mid:<8} S={S} vế trái={100 * S * mid + 100 * dist[-1]:<12} vế phải={H * mid:<12} ok={r}")
        if r: hi = mid
        else: lo = mid + 1
    print("=> v* =", lo, f"(tổng {step} bước)")
    return lo

assert trace_bs([1, 3, 2], 2.7) == 3

bước  1: lo=1        hi=10000000 mid=5000000  S=2 vế trái=1000000200   vế phải=1350000000   ok=True
bước  2: lo=1        hi=5000000  mid=2500000  S=2 vế trái=500000200    vế phải=675000000    ok=True
bước  3: lo=1        hi=2500000  mid=1250000  S=2 vế trái=250000200    vế phải=337500000    ok=True
bước  4: lo=1        hi=1250000  mid=625000   S=2 vế trái=125000200    vế phải=168750000    ok=True
bước  5: lo=1        hi=625000   mid=312500   S=2 vế trái=62500200     vế phải=84375000     ok=True
bước  6: lo=1        hi=312500   mid=156250   S=2 vế trái=31250200     vế phải=42187500     ok=True
bước 22: lo=1        hi=5        mid=3        S=2 vế trái=800          vế phải=810          ok=True
bước 23: lo=1        hi=3        mid=2        S=3 vế trái=800          vế phải=540          ok=False
=> v* = 3 (tổng 23 bước)


| giai đoạn | lo | hi | mid | ok(mid) |
|---|---|---|---|---|
| đầu | 1 | 10⁷ | 5 000 000 | ✓ ⇒ hi = mid |
| … | 1 | … | … | ✓ liên tục, `hi` giảm một nửa |
| gần cuối | 1 | 4 | 2 | ✗ (`T(2) = 4`) ⇒ lo = 3 |
| cuối | 3 | 4 | 3 | ✓ ⇒ hi = 3 ⇒ dừng, `v* = 3` |

Khoảng 24 bước cho `10⁷` — mỗi bước một lần `O(n)`.

## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 2** — binary search trên `[1, 10⁷]`, kiểm tra bằng số nguyên.

In [13]:
class Solution:
    def minSpeedOnTime(self, dist: List[int], hour: float) -> int:
        n = len(dist)
        H = round(hour * 100)                       # hour tính bằng phần trăm giờ (số nguyên)
        if H <= 100 * (n - 1):
            return -1
        head, last = dist[:-1], dist[-1]

        def ok(v: int) -> bool:
            s = 0
            for d in head:
                s += (d + v - 1) // v
            return 100 * s * v + 100 * last <= H * v

        lo, hi = 1, 10 ** 7
        if not ok(hi):
            return -1
        while lo < hi:
            mid = (lo + hi) // 2
            if ok(mid):
                hi = mid
            else:
                lo = mid + 1
        return lo

for (dist, hour), exp in examples:
    got = Solution().minSpeedOnTime(dist, hour); print(got, exp); assert got == exp

1 1
3 3
-1 -1


## 10. Vì sao thuật toán đúng

**Mô hình thời gian.** Bắt đầu ở mốc 0 (nguyên). Sau chuyến `i < n−1` xuất phát ở mốc nguyên `t`, ta đến ở `t + d/v` và chờ đến mốc nguyên kế tiếp `≥` thời điểm đó, tức `t + ⌈d/v⌉`. Chuyến cuối không phải chờ. Quy nạp ⇒ thời điểm đến `T(v) = Σ_{i<n−1} ⌈dᵢ/v⌉ + d_last/v`.

**Đơn điệu.** `v₁ < v₂` ⇒ `d/v₁ > d/v₂` và `⌈d/v₁⌉ ≥ ⌈d/v₂⌉` ⇒ `T(v₁) ≥ T(v₂)`. Vậy `ok` đơn điệu và binary search trả `v` nhỏ nhất với `ok(v)`.

**Kiểm tra số nguyên chính xác.** `hour = H/100` chính xác (tối đa hai chữ số thập phân; `round` loại sai số khi đọc `hour·100`). Nhân hai vế `S + last/v ≤ H/100` với `100v > 0` giữ chiều bất đẳng thức ⇒ `100Sv + 100·last ≤ Hv`.

**Bất khả thi.** `T(v) > n − 1` với mọi `v` (mỗi chuyến đầu `≥ 1`, chuyến cuối `> 0`) ⇒ nếu `hour ≤ n − 1` thì không có đáp án. Ngược lại, Cách 3 chỉ ra `ok(hi)` đúng với `hi` hữu hạn ⇒ có đáp án; đề đảm bảo nó `≤ 10⁷` ⇒ kiểm tra `ok(10⁷)` là an toàn. ∎

### 10.1 Kiểm chứng thực nghiệm tính đơn điệu và mô hình thời gian

Mô phỏng hành trình bằng `Fraction` (đi, chờ đến mốc nguyên, đi tiếp) và so với công thức `T(v)`; đồng thời kiểm tra `T` không tăng theo `v`.

In [14]:
def simulate(dist, v):
    t = Fraction(0)
    for i, d in enumerate(dist):
        t += Fraction(d, v)
        if i < len(dist) - 1:
            t = Fraction(math.ceil(t))            # chờ tới giờ nguyên
    return t

rng = random.Random(1010)
for _ in range(500):
    dist = [rng.randint(1, 30) for _ in range(rng.randint(1, 6))]
    Ts = [T_exact(dist, v) for v in range(1, 40)]
    assert all(simulate(dist, v) == Ts[v - 1] for v in range(1, 40))
    assert all(Ts[k] >= Ts[k + 1] for k in range(len(Ts) - 1))
    assert all(t > len(dist) - 1 for t in Ts)
print("Mô phỏng = công thức T(v); T không tăng; T > n − 1 (500 mảng x 39 tốc độ)")

Mô phỏng = công thức T(v); T không tăng; T > n − 1 (500 mảng x 39 tốc độ)


## 11. Phân tích Time / Space Complexity

Gọi `n ≤ 10⁵`, `M = 10⁷`.

| Cách | Time | Space |
|---|---|---|
| 1. Quét tuyến tính `v` | `O(v* · n)` | `O(1)` |
| 2. Binary search `[1, 10⁷]` | `O(n log M)` ≈ `24n` | `O(n)` (lát cắt) |
| 3. Binary search cận chặt | `O(n log hi)` | `O(n)` |
| 4. `bisect` + `key` | `O(n log M)` | `O(n)` |

In [15]:
rng = random.Random(1870)
big = [rng.randint(1, 10 ** 5) for _ in range(10 ** 5)]
for hour in (10 ** 5 - 1 + 0.01, 2.5 * 10 ** 5, 10 ** 9):
    for name, f in [("bs", Solution().minSpeedOnTime), ("tight", minspeed_tight), ("bisect", minspeed_bisect)]:
        t = time.perf_counter(); r = f(big, hour); print(f"hour={hour:<12} {name:<6}: v*={r:<9} ({(time.perf_counter() - t) * 1000:.0f} ms)")

hour=99999.01     bs    : v*=2744400   (93 ms)


hour=99999.01     tight : v*=2744400   (87 ms)
hour=99999.01     bisect: v*=2744400   (97 ms)


hour=250000.0     bs    : v*=24970     (95 ms)


hour=250000.0     tight : v*=24970     (66 ms)
hour=250000.0     bisect: v*=24970     (99 ms)


hour=1000000000   bs    : v*=5         (100 ms)


hour=1000000000   tight : v*=5         (78 ms)
hour=1000000000   bisect: v*=5         (123 ms)


## 12. Bộ test

In [16]:
ALL = {"bs": minspeed_bs, "tight": minspeed_tight, "bisect": minspeed_bisect, "submit": Solution().minSpeedOnTime}
def run_all(dist, hour, brute=True):
    res = {k: f(list(dist), hour) for k, f in ALL.items()}
    if brute:
        res["brute"] = minspeed_bruteforce(dist, hour, limit=3000)
    assert len(set(res.values())) == 1, (dist, hour, res)
    return res["submit"]

for (dist, hour), exp in examples:
    assert run_all(dist, hour) == exp
# ví dụ tự dựng (đối chiếu lại đề gốc) — kiểm bằng brute force (Fraction) trong run_all
edge = [(([5], 1), 5), (([1, 1], 1.0), -1), (([2, 2], 3), 2), (([1, 1, 1], 2.5), 2),
        (([1, 14], 1.14), 100), (([1], 1), 1), (([3], 0.01 + 1), 3), (([1, 1, 3], 2.01), 300)]
for (dist, hour), exp in edge:
    assert run_all(dist, hour) == exp, (dist, hour, run_all(dist, hour))
rng = random.Random(18700)
for _ in range(400):
    n = rng.randint(1, 6); dist = [rng.randint(1, 30) for _ in range(n)]
    H = rng.randint(100, 100 * (n + 8)); run_all(dist, H / 100)
assert Solution().minSpeedOnTime([1, 1, 10 ** 5], 2.01) == 10 ** 7          # đáp án chạm đúng cận 10^7
for _ in range(5):
    n = rng.randint(1, 10 ** 4); dist = [rng.randint(1, 10 ** 5) for _ in range(n)]
    hour = round(rng.uniform(n - 1 + 0.01, 3 * n), 2)
    run_all(dist, hour, brute=False)
print("Ví dụ, 8 edge case, 400 test nhỏ (đối chiếu Fraction), cận 10^7, 5 test lớn: OK")

Ví dụ, 8 edge case, 400 test nhỏ (đối chiếu Fraction), cận 10^7, 5 test lớn: OK


### 12.1 Kiểm tra tính chất

- Tăng `hour` không làm đáp án tăng (nhiều thời gian hơn ⇒ tốc độ tối thiểu không lớn hơn).
- Với `v*` trả về: `T(v*) ≤ hour` và (nếu `v* > 1`) `T(v* − 1) > hour` — kiểm bằng `Fraction`.
- Nhân mọi `dist` với `k` ⇒ nếu `v*` cũ là đáp án, `k·v*` vẫn đến kịp (đáp án mới `≤ k·v*`).

In [17]:
rng = random.Random(121)
f = Solution().minSpeedOnTime
for _ in range(500):
    n = rng.randint(1, 6); dist = [rng.randint(1, 50) for _ in range(n)]
    H = rng.randint(100 * (n - 1) + 1, 100 * (n + 10)); h = Fraction(H, 100)
    v = f(dist, H / 100)
    assert v != -1
    assert T_exact(dist, v) <= h and (v == 1 or T_exact(dist, v - 1) > h)
    assert f(dist, (H + rng.randint(1, 50)) / 100) <= v
    k = rng.randint(2, 4)
    assert f([k * d for d in dist], H / 100) <= k * v
print("3 tính chất đúng trên 500 test")

3 tính chất đúng trên 500 test


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Làm tròn lên cả chuyến cuối

In [18]:
def wrong_ceil_last(dist, hour):
    def ok(v):
        return sum(math.ceil(d / v) for d in dist) <= hour        # SAI: chuyến cuối không phải chờ
    lo, hi = 1, 10 ** 7
    if not ok(hi):
        return -1
    while lo < hi:
        mid = (lo + hi) // 2
        if ok(mid): hi = mid
        else: lo = mid + 1
    return lo

for (dist, hour), exp in examples:
    print(f"hour={hour}: sai {wrong_ceil_last(dist, hour)} | đúng {exp}")

hour=6: sai 1 | đúng 1
hour=2.7: sai -1 | đúng 3
hour=1.9: sai -1 | đúng -1


**Vì sao sai:** đến đích là xong, không phải chờ tới giờ nguyên. Với `hour = 2.7`, mọi tốc độ đều cho tổng làm tròn `≥ 3` ⇒ báo `−1`. **Sửa:** chuyến cuối dùng `d/v` thật.

### Lỗi 2 — So sánh bằng số thực (sai ở đúng biên)

In [19]:
def float_version(dist, hour):
    def ok(v):
        return sum(math.ceil(d / v) for d in dist[:-1]) + dist[-1] / v <= hour   # sai số dấu phẩy động
    if hour <= len(dist) - 1:
        return -1
    lo, hi = 1, 10 ** 7
    if not ok(hi):
        return -1
    while lo < hi:
        mid = (lo + hi) // 2
        if ok(mid): hi = mid
        else: lo = mid + 1
    return lo

dist, hour = [1, 14], 1.14          # ví dụ tự dựng (đối chiếu lại đề gốc)
print(f"1 + 14/100 = {1 + 14 / 100!r} > {hour!r}")
print("số thực:", float_version(dist, hour), "| số nguyên:", Solution().minSpeedOnTime(dist, hour),
      "| Fraction:", minspeed_bruteforce(dist, hour, limit=500))

1 + 14/100 = 1.1400000000000001 > 1.14
số thực: 101 | số nguyên: 100 | Fraction: 100


**Vì sao sai:** ở `v = 100`, thời gian thật đúng bằng `1.14`, nhưng `1 + 0.14` trong số thực là `1.1400000000000001 > 1.14` ⇒ báo không kịp ⇒ trả `101`. **Sửa:** so sánh bằng số nguyên (`H = round(hour·100)`) hoặc ít nhất thêm `ε` nhỏ (cách sau dễ chọn `ε` sai).

### Lỗi 3 — Không kiểm tra bất khả thi

In [20]:
def wrong_no_impossible(dist, hour):
    ok, H = make_ok(dist, hour)
    return first_true(1, 10 ** 7, ok)                               # SAI: giả định ok(10^7) đúng

for (dist, hour), exp in examples:
    print(f"hour={hour}: sai {wrong_no_impossible(dist, hour)} | đúng {exp}")

hour=6: sai 1 | đúng 1
hour=2.7: sai 3 | đúng 3
hour=1.9: sai 10000000 | đúng -1


**Vì sao sai:** khi không tốc độ nào đủ, binary search vẫn "hội tụ" về `hi = 10⁷`. **Sửa:** kiểm tra `hour > n − 1` (hoặc `ok(hi)`) trước.

### Lỗi 4 — Cận trên quá nhỏ (ví dụ `max(dist)`)

In [21]:
def wrong_small_hi(dist, hour):
    ok, H = make_ok(dist, hour)
    if H <= 100 * (len(dist) - 1):
        return -1
    hi = max(dist)                                                  # SAI: có thể cần v > max(dist)
    if not ok(hi):
        return -1
    return first_true(1, hi, ok)

dist, hour = [1, 1, 3], 2.01        # ví dụ tự dựng (đối chiếu lại đề gốc)
print("sai :", wrong_small_hi(dist, hour), "| đúng:", Solution().minSpeedOnTime(dist, hour), "| brute:", minspeed_bruteforce(dist, hour, limit=1000))

sai : -1 | đúng: 300 | brute: 300


**Vì sao sai:** khi `hour − (n−1)` rất nhỏ (ở đây 0.01), chuyến cuối phải cực nhanh: `3/v ≤ 0.01` ⇒ `v ≥ 300 > max(dist)`. **Sửa:** dùng `10⁷` theo đề hoặc cận chặt của Cách 3.

### Lỗi 5 — Làm tròn lên bằng `int(d / v) + 1`

In [22]:
def wrong_ceil(dist, hour):
    H = round(hour * 100)
    if H <= 100 * (len(dist) - 1):
        return -1
    def ok(v):
        S = sum(int(d / v) + 1 for d in dist[:-1])                  # SAI khi d chia hết cho v
        return 100 * S * v + 100 * dist[-1] <= H * v
    if not ok(10 ** 7):
        return -1
    return first_true(1, 10 ** 7, ok)

for (dist, hour), exp in examples:
    print(f"hour={hour}: sai {wrong_ceil(dist, hour)} | đúng {exp}")

hour=6: sai 2 | đúng 1
hour=2.7: sai 4 | đúng 3
hour=1.9: sai -1 | đúng -1


**Vì sao sai:** `int(3/3) + 1 = 2` trong khi `⌈3/3⌉ = 1`: khi đến **đúng** giờ nguyên thì không phải chờ (Ví dụ 1 nói rõ). **Sửa:** `(d + v − 1) // v`.

### Lỗi 6 — Quét tuyến tính tốc độ

In [23]:
dist = [rng.randint(1, 10 ** 5) for _ in range(2000)]
hour = 2000 - 1 + 0.5
t = time.perf_counter(); r1 = Solution().minSpeedOnTime(dist, hour); t1 = time.perf_counter() - t
ok, _ = make_ok(dist, hour)
t = time.perf_counter(); v = 1
while not ok(v) and v < 3000:
    v += 1
t2 = time.perf_counter() - t
print(f"binary search: v* = {r1} ({t1 * 1000:.0f} ms) | quét tuyến tính 3000 tốc độ đầu ({t2 * 1000:.0f} ms) mà mới tới v = {v}")

binary search: v* = 189904 (4 ms) | quét tuyến tính 3000 tốc độ đầu (293 ms) mà mới tới v = 3000


**Vì sao không tốt:** đáp án có thể tới `10⁷`; quét từng `v` với `O(n)` mỗi lần là `10¹²`. **Sửa:** binary search.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tìm **giá trị nhỏ nhất/lớn nhất** sao cho điều kiện **đơn điệu** thỏa" ⇒ binary search trên đáp án (875 Koko, 1011 Ship Packages, 1482, 2187).
- Viết riêng hàm `ok(v)` `O(n)` rồi mới viết binary search.
- Đầu vào số thực có số chữ số thập phân cố định ⇒ **nhân lên thành số nguyên**.
- Kiểm tra biên: `hour ≤ n − 1` ⇒ `−1`; đáp án có thể **đúng bằng** cận trên `10⁷`.
- Làm tròn lên số nguyên: `(a + b − 1) // b`.

## 15. Biến thể của bài

1. **875 Koko Eating Bananas**: `Σ ⌈pile/k⌉ ≤ h` — giống hệt nhưng mọi số hạng đều làm tròn.
2. **1011 Capacity To Ship Packages Within D Days**: tham lam trong `ok`, binary search sức chứa (Bài tập 4).
3. **Tàu khởi hành ở bội số của `p` giờ** thay vì mỗi giờ (Bài tập 5).
4. **2187 Minimum Time to Complete Trips**: binary search trên thời gian, `ok(t) = Σ t // time[i] ≥ totalTrips`.
5. **Mỗi chuyến có tốc độ riêng** nhưng phải chọn **một hệ số chung** — cùng khung `ok`.

### 15.1 Code biến thể: 875 Koko Eating Bananas, đối chiếu quét tuyến tính

In [24]:
def min_eating_speed(piles, h):
    ok = lambda k: sum((p + k - 1) // k for p in piles) <= h
    return first_true(1, max(piles), ok)

rng = random.Random(875)
for _ in range(300):
    piles = [rng.randint(1, 40) for _ in range(rng.randint(1, 6))]
    h = rng.randint(len(piles), 60)
    lin = next(k for k in range(1, 41) if sum(math.ceil(p / k) for p in piles) <= h)
    assert min_eating_speed(piles, h) == lin
print("875 khớp quét tuyến tính;", min_eating_speed([3, 6, 7, 11], 8), "(LeetCode 875 ví dụ: 4)")

875 khớp quét tuyến tính; 4 (LeetCode 875 ví dụ: 4)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét `v` tăng dần | Thử từng tốc độ | `O(v*·n)` | `O(1)` | Kiểm chứng |
| 2. Binary search `[1,10⁷]` | `ok` số nguyên | `O(n log M)` | `O(n)` | **Bản nộp** |
| 3. Cận chặt | `hi` từ `max(dist)` và slack | `O(n log hi)` | `O(n)` | Không phụ thuộc bảo đảm `10⁷` |
| 4. `bisect` + `key` | Thư viện chuẩn | `O(n log M)` | `O(n)` | Viết nhanh (Python ≥ 3.10) |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `([5], 1)`; (b) `([1,1], 1.0)`; (c) `([2,2], 3)`; (d) `([1,1,1], 2.5)`.

**Bài 2 (tính tay):** Với `dist = [1,3,2]`, `hour = 6`, binary search trên `[1, 8]`: liệt kê các `mid` được thử và kết quả `ok`.

**Bài 3 (điền TODO):** Hoàn thiện `ok_todo(dist, H, v)` bằng số nguyên.

**Bài 4 (tự viết lại):** Viết `ship_within_days(weights, days)` (LeetCode 1011): sức chứa nhỏ nhất để chở hết theo thứ tự trong `days` ngày; đối chiếu quét tuyến tính.

**Bài 5 (mini challenge):** Viết `min_speed_period(dist, hour, p)`: tàu chỉ khởi hành ở các mốc **bội của `p` giờ** (`p` nguyên dương, xuất phát ở mốc 0). Dùng số nguyên tuyệt đối; đối chiếu mô phỏng bằng `Fraction`.

In [25]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def ok_todo(dist, H, v):
    # TODO: S = tổng (d + v - 1) // v cho mọi chuyến trừ chuyến cuối
    #       trả về 100 * S * v + 100 * dist[-1] <= H * v
    pass

In [26]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def ship_within_days(weights, days):
    # TODO: ok(c): tham lam xếp hàng vào ngày hiện tại khi còn chỗ; binary search c trên [max(w), sum(w)]
    pass

def min_speed_period(dist, hour, p):
    # TODO: chuyến không-cuối chiếm p * ceil(d / (v * p)) giờ; chuyến cuối d / v
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: kiểm tra `hour > n − 1`. Mức 2: thử `v` nhỏ. Mức 3: (a) `5`; (b) `−1`; (c) `2` (`v = 1`: 2 + 2 = 4 > 3; `v = 2`: 1 + 1 = 2); (d) `2` (`v = 2`: 1 + 1 + 0.5 = 2.5).

**Bài 2.** Mức 1: `lo = 1, hi = 8`. Mức 2: `mid = 4` ✓, `mid = 2` ✓, `mid = 1` ✓ (`T(1) = 6`). Mức 3: đáp án `1` sau 3 bước.

**Bài 3.** Mức 1: `H` đã là phần trăm giờ. Mức 2: làm tròn lên bằng số nguyên. Mức 3: nhân chéo với `100·v`.

**Bài 4.** Mức 1: `ok` đơn điệu theo sức chứa. Mức 2: `lo = max(weights)` (mỗi kiện phải vừa). Mức 3: đếm số ngày bằng tham lam, so với `days`.

**Bài 5.** Mức 1: sau chuyến không-cuối, chờ tới bội kế tiếp của `p`. Mức 2: vì xuất phát ở bội của `p`, chuyến chiếm `p·⌈d/(v·p)⌉`. Mức 3: điều kiện `100·p·S·v + 100·last ≤ H·v`; bất khả thi khi `H ≤ 100·p·(n−1)`.

## 19. Đáp án tham khảo cho bài tập

In [27]:
# Bài 1, 2, 3
f = Solution().minSpeedOnTime
b1 = [f(d, h) for d, h in [([5], 1), ([1, 1], 1.0), ([2, 2], 3), ([1, 1, 1], 2.5)]]
print("Bài 1:", b1); assert b1 == [5, -1, 2, 2]
print("Bài 2:"); assert trace_bs([1, 3, 2], 6, hi=8) == 1
def ok_todo(dist, H, v):
    S = sum((d + v - 1) // v for d in dist[:-1])
    return 100 * S * v + 100 * dist[-1] <= H * v
rng = random.Random(3)
for _ in range(300):
    dist = [rng.randint(1, 20) for _ in range(rng.randint(1, 5))]; H = rng.randint(100, 900); v = rng.randint(1, 30)
    assert ok_todo(dist, H, v) == (T_exact(dist, v) <= Fraction(H, 100))
print("Bài 3: OK")

Bài 1: [5, -1, 2, 2]
Bài 2:
bước  1: lo=1        hi=8        mid=4        S=2 vế trái=1000         vế phải=2400         ok=True
bước  2: lo=1        hi=4        mid=2        S=3 vế trái=800          vế phải=1200         ok=True
bước  3: lo=1        hi=2        mid=1        S=4 vế trái=600          vế phải=600          ok=True
=> v* = 1 (tổng 3 bước)
Bài 3: OK


In [28]:
# Bài 4, 5
def ship_within_days(weights, days):
    def ok(c):
        need, cur = 1, 0
        for w in weights:
            if cur + w > c:
                need += 1; cur = 0
            cur += w
        return need <= days
    return first_true(max(weights), sum(weights), ok)

def min_speed_period(dist, hour, p):
    H, n = round(hour * 100), len(dist)
    if H <= 100 * p * (n - 1):
        return -1
    def ok(v):
        S = sum((d + v * p - 1) // (v * p) for d in dist[:-1])
        return 100 * p * S * v + 100 * dist[-1] <= H * v
    hi = max(max(dist), -(-100 * dist[-1] // (H - 100 * p * (n - 1))))
    return first_true(1, hi, ok)

def period_brute(dist, hour, p, limit=3000):
    h = Fraction(round(hour * 100), 100)
    if h <= p * (len(dist) - 1):
        return -1                                  # mọi chuyến không-cuối chiếm ít nhất p giờ
    for v in range(1, limit + 1):
        t = Fraction(0)
        for i, d in enumerate(dist):
            t += Fraction(d, v)
            if i < len(dist) - 1:
                t = p * Fraction(math.ceil(t / p))
        if t <= h:
            return v
    return -1

rng = random.Random(5)
for _ in range(300):
    w = [rng.randint(1, 10) for _ in range(rng.randint(1, 7))]; d = rng.randint(1, len(w))
    def days_needed(c):
        need, cur = 1, 0
        for x in w:
            if cur + x > c: need += 1; cur = 0
            cur += x
        return need
    assert ship_within_days(w, d) == next(c for c in range(max(w), sum(w) + 1) if days_needed(c) <= d)
    dist = [rng.randint(1, 15) for _ in range(rng.randint(1, 4))]; p = rng.randint(1, 3)
    hour = rng.randint(100, 100 * (p * len(dist) + 4)) / 100
    assert min_speed_period(dist, hour, p) == period_brute(dist, hour, p)
print("Bài 4, 5: OK |", ship_within_days([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 5), "(1011: 15) |", min_speed_period([1, 3, 2], 6, 2))

Bài 4, 5: OK | 15 (1011: 15) | 2


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (binary search trên đáp án + số nguyên tuyệt đối):**

```python
H = round(hour * 100)                       # phần trăm giờ
if H <= 100 * (n - 1): return -1
def ok(v):
    S = sum((d + v - 1) // v for d in dist[:-1])
    return 100 * S * v + 100 * dist[-1] <= H * v
lo, hi = 1, 10 ** 7
if not ok(hi): return -1
while lo < hi:
    mid = (lo + hi) // 2
    if ok(mid): hi = mid
    else: lo = mid + 1
return lo
```

**Checklist:**
- [ ] Chuyến cuối **không** làm tròn lên?
- [ ] Làm tròn lên bằng `(d + v − 1) // v` (không phải `int(d/v) + 1`)?
- [ ] So sánh bằng **số nguyên** (không sai số thực ở biên như `1.14`)?
- [ ] Kiểm tra `−1` khi `hour ≤ n − 1`?
- [ ] Cận trên đủ lớn (`10⁷`, đáp án có thể chạm đúng cận)?